In [ ]:
#upload the raw training corpus
from google.colab import files
uploaded = files.upload()

Saving Avengers.Infinity.War.txt to Avengers.Infinity.War.txt


Data Cleaning

In [ ]:
#Clean the raw training corpus
import re
import string
#Read the subtitle file
with open("Training.txt", "r", encoding="utf-8") as f:
    text = f.read()
#Lowercase
text = text.lower()
#Remove timestamps
text = re.sub(r'\d+:\d+:\d+,\d+ --> \d+:\d+:\d+,\d+', '', text)
#Remove numbers
text = re.sub(r'\d+', '', text)
#Remove punctuation
text = text.translate(str.maketrans('', '', string.punctuation))
#Split into words
words = text.split()
#Save cleaned output
with open("clean_data.txt", "w", encoding="utf-8") as f:
    f.write(" ".join(words))

print("Cleaned data saved to clean_data.txt")

Cleaned data saved to clean_data.txt


In [ ]:
#Make the Vocabulary

vocabulary = set(words)
vocab_size = len(vocabulary)

print("Vocabulary size:", vocab_size)

Vocabulary size: 1647


In [ ]:
#Count the frequency of each word in the training corpus

from collections import Counter

word_freq = Counter(words)
total_words = len(words)


Language Model Generation

In [ ]:
#Save unigram model
with open("unigram_model.txt", "w", encoding="utf-8") as f:
    for word, prob in unigram_prob.items():
        f.write(f"{word}\t{prob}\n")

print("Unigram model saved to unigram_model.txt")

#Unigram probability
unigram_prob = {word: count/total_words for word, count in word_freq.items()}

Unigram model saved to unigram_model.txt


In [ ]:
#Save bigram model
from collections import defaultdict

bigram_counts = defaultdict(int)

for i in range(len(words)-1):
    bigram = (words[i], words[i+1])
    bigram_counts[bigram] += 1

# save bigram model
with open("bigram_model.txt", "w", encoding="utf-8") as f:
    for (w1, w2), count in bigram_counts.items():
        f.write(f"{w1} {w2}\t{count}\n")

print("Bigram model saved to bigram_model.txt")


#Bigram probability method
def bigram_probability(prev_word, word):

    bigram_count = bigram_counts.get((prev_word, word), 0)
    prev_count = word_freq.get(prev_word, 1)

    return bigram_count / prev_count

Bigram model saved to bigram_model.txt


Error Model Generation/Implementation

In [ ]:
!pip install textdistance
#Calculating edit distance

import textdistance

def candidates(word):
    return [v for v in vocabulary if textdistance.levenshtein(word, v) <= 1]

Imeplement Noisy Channel

In [ ]:
# Using edit distance
def error_probability(typo, word):
    distance = textdistance.levenshtein(typo, word)
    return 1/(distance+1)

In [ ]:
# Corrects a misspelled word using unigram word probability and edit distance error probability

def correct_word_unigram(typo):

    candidate_words = candidates(typo)

    if not candidate_words:
        return typo

    best_word = typo
    best_score = 0

    for word in candidate_words:

        p_word = unigram_prob.get(word, 0)
        p_typo = error_probability(typo, word)

        score = p_word * p_typo

        if score > best_score:
            best_score = score
            best_word = word

    return best_word

In [ ]:
# Corrects a misspelled word using bigram word probability and edit distance error probability

def correct_word_bigram(prev_word, typo):

    candidate_words = candidates(typo)

    if not candidate_words:
        return typo

    best_word = typo
    best_score = 0

    for word in candidate_words:

        p_bigram = bigram_probability(prev_word, word)
        p_typo = error_probability(typo, word)

        score = p_bigram * p_typo

        if score > best_score:
            best_score = score
            best_word = word

    return best_word

In [ ]:
# Entire sentence word by word using a unigram spell checker
def correct_sentence_unigram(sentence):

    words = sentence.split()
    corrected = []

    for word in words:
        corrected.append(correct_word_unigram(word))

    return " ".join(corrected)

In [ ]:
# Corrects an entire sentence word by word using a bigram spell checker

def correct_sentence_bigram(sentence):

    words = sentence.split()
    corrected = [words[0]]

    for i in range(1, len(words)):

        prev_word = corrected[i-1]
        corrected_word = correct_word_bigram(prev_word, words[i])

        corrected.append(corrected_word)

    return " ".join(corrected)

Test Corpus

In [ ]:
# Test corpus

test_sentences = [
    "i dont no what you doing", "this movi is amazng", "we shold go hom now",
    "she dont like this movi", "he is a gud frend", "this book is very intresting",
    "i realy like this song", "it is a privlege to be saved", "saved by the great titan",
    "i cant find my phon", "the asgardian vessel is here", "we wil meat tomorow",
    "i hav a qustion", "this is not a warcraft", "he didnt recive the mesage",
    "please chek the answr", "it is salvation", "i am the son of odin",
    "the infinity stons are lost", "the tesseract or your brothers hed", "i dont undrstand this topic",
    "he is wating for bus", "we shold finish the work", "the scale tips toward balanse",
    "she likes to red books", "i forgot my passwrd", "this storry is very sad",
    "we need mor practce", "he is runing very fast", "the chld is cryng",
    "destiny arrives all the same", "half the universe dies", "he is wrting a letter",
    "we wil start the class soon", "she baught a new dress", "the sun will shine on us agan",
    "this song is very populer", "he drived very fast", "she is cleening the room",
    "you think this is sufering", "because of your sacrifice", "i am tring my best",
    "this topic is very imprtant", "he has two infinity stons", "she is listning to music",
    "get this man a sheild", "this problem is too hardd", "i am readng a novel",
    "she is wachng tv", "universal scales tip toward balanse"
]

correct_sentences = [
    "i dont know what you doing", "this movie is amazing", "we should go home now",
    "she dont like this movie", "he is a good friend", "this book is very interesting",
    "i really like this song", "it is a privilege to be saved", "saved by the great titan",
    "i cant find my phone", "the asgardian vessel is here", "we will meet tomorrow",
    "i have a question", "this is not a warcraft", "he didnt receive the message",
    "please check the answer", "it is salvation", "i am the son of odin",
    "the infinity stones are lost", "the tesseract or your brothers head", "i dont understand this topic",
    "he is waiting for bus", "we should finish the work", "the scale tips toward balance",
    "she likes to read books", "i forgot my password", "this story is very sad",
    "we need more practice", "he is running very fast", "the child is crying",
    "destiny arrives all the same", "half the universe dies", "he is writing a letter",
    "we will start the class soon", "she bought a new dress", "the sun will shine on us again",
    "this song is very popular", "he drove very fast", "she is cleaning the room",
    "you think this is suffering", "because of your sacrifice", "i am trying my best",
    "this topic is very important", "he has two infinity stones", "she is listening to music",
    "get this man a shield", "this problem is too hard", "i am reading a novel",
    "she is watching tv", "universal scales tip toward balance"
]

In [ ]:
#test the test corpus using both unigram and bigram models

for sentence in test_sentences:

    print("Input:", sentence)

    print("Unigram:", correct_sentence_unigram(sentence))

    print("Bigram:", correct_sentence_bigram(sentence))

    print()

Input: i dont no what you doing
Unigram: i dont to what you doing
Bigram: i dont do that you doing

Input: this movi is amazng
Unigram: this movie i amazing
Bigram: this movi is amazing

Input: we shold go hom now
Unigram: we should to him no
Bigram: we should go home now

Input: she dont like this movi
Unigram: the dont like this movie
Bigram: she dont like this movi

Input: he is a gud frend
Unigram: the i a god friend
Bigram: he is a god frend

Input: this book is very intresting
Unigram: this look i very intresting
Bigram: this book is very intresting

Input: i realy like this song
Unigram: i really like this long
Bigram: i really like this song

Input: it is a privlege to be saved
Unigram: it i a privilege to we saved
Bigram: it is a privlege to be saved

Input: saved by the great titan
Unigram: saved my the great titan
Bigram: saved by the great titan

Input: i cant find my phon
Unigram: i cant find my phone
Bigram: i cant find my phon

Input: the asgardian vessel is here
Unigram

calculate the Precision, Recall, and F1-score.

In [ ]:
#Evaluate the unigram model using precision_score , recall_score, f1_score
from sklearn.metrics import precision_score, recall_score, f1_score
y_true = []
y_pred = []
for i in range(len(test_sentences)):
    predicted = correct_sentence_unigram(test_sentences[i]).split()
    actual = correct_sentences[i].split()
    for p,a in zip(predicted, actual):
        y_pred.append(p)
        y_true.append(a)

precision = precision_score(y_true, y_pred, average='micro')
recall = recall_score(y_true, y_pred, average='micro')
f1 = f1_score(y_true, y_pred, average='micro')

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Precision: 0.6244897959183674
Recall: 0.6244897959183674
F1 Score: 0.6244897959183674


In [ ]:
#Evaluate the bigram model using precision_score , recall_score, f1_score
from sklearn.metrics import precision_score, recall_score, f1_score
y_true = []
y_pred = []
for i in range(len(test_sentences)):
    predicted = correct_sentence_bigram(test_sentences[i]).split()
    actual = correct_sentences[i].split()
    for p,a in zip(predicted, actual):
        y_pred.append(p)
        y_true.append(a)

precision = precision_score(y_true, y_pred, average='micro')
recall = recall_score(y_true, y_pred, average='micro')
f1 = f1_score(y_true, y_pred, average='micro')

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Precision: 0.8204081632653061
Recall: 0.8204081632653061
F1 Score: 0.8204081632653061


Time Complexity Analysis

In [ ]:
#Measure the unigram model time complexity
import time
start = time.time()
for sentence in test_sentences:
    correct_sentence_unigram(sentence)
end = time.time()
unigram_time = end - start

print("Unigram Time:", unigram_time)
avg_unigram_time = unigram_time / len(test_sentences)
print("Average Unigram Time:", avg_unigram_time)

Unigram Time: 14.072590827941895
Average Unigram Time: 0.2814518165588379


In [ ]:
#Measure the bigram model time complexity
start = time.time()
for sentence in test_sentences:
    correct_sentence_bigram(sentence)
end = time.time()
bigram_time = end - start
print("Bigram Time:", bigram_time)
avg_bigram_time = bigram_time / len(test_sentences)
print("Average Bigram Time:", avg_bigram_time)

Bigram Time: 11.383132696151733
Average Bigram Time: 0.22766265392303467
